# Colab training notebook for Industrial Defects

This notebook clones the repository, sets up a Colab-compatible environment, prepares the MVTec AD dataset, and runs the reusable cloud training pipeline.

In [ ]:
# Clone the repository into Colab workspace
import os
import shutil
import sys
from pathlib import Path

repo_dir = Path('/content/Industrial_defects')
if not repo_dir.exists():
    !git clone https://github.com/your-org/Industrial_defects.git /content/Industrial_defects
else:
    print('Repository already present at /content/Industrial_defects')

os.chdir(repo_dir)
print('Working directory:', repo_dir)

In [ ]:
# Install Colab-compatible runtime dependencies only
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q numpy pillow scikit-learn pyyaml fastapi uvicorn[standard] python-multipart mlflow sqlalchemy albumentations opencv-python-headless tqdm

In [ ]:
# Prepare dataset location outside Git
data_dir = Path('/content/data')
data_dir.mkdir(parents=True, exist_ok=True)

mvtec_dir = data_dir / 'mvtec_ad'
if not mvtec_dir.exists():
    print('Please download the MVTec AD dataset into /content/data/mvtec_ad before running the training cell.')
else:
    print('Dataset found at', mvtec_dir)

In [ ]:
# Detect GPU and configure torch
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('CUDA available:', torch.cuda.is_available())
print('Device:', device)

In [ ]:
# Run the reusable training pipeline
from src.training.cloud_training import run_cloud_training

result = run_cloud_training(
    config_path='configs/cloud_training.yaml',
    dataset_path='/content/data/mvtec_ad',
)
result

In [ ]:
# Zip artifacts for download
import shutil
from pathlib import Path

artifacts_dir = Path('artifacts/cloud_training')
archive_path = Path('/content/cloud_training_artifacts.zip')
if artifacts_dir.exists():
    shutil.make_archive('/content/cloud_training_artifacts', 'zip', root_dir='.', base_dir='artifacts/cloud_training')
    print('Artifacts archived to', archive_path)
else:
    print('Artifacts directory not found')